# Part 3 · Sign up, sign in with JWT, data isolation and human approval

Many users share one agent and one checkpoint database. This part adds accounts to the service and the two checks that
run before any route: **who is calling** (a signed access token) and **is this thread theirs** (ownership). Then a
paused run can be resumed over HTTP by its owner only.

| File | Role |
| --- | --- |
| [`routers/auth.py`](../app/routers/auth.py) | `POST /auth/signup`, `POST /auth/token` (sign in), `GET /auth/me` |
| [`core/security.py`](../app/core/security.py) | Argon2 password hashes, JWT create/verify, `current_user` (401 otherwise) |
| [`services/threads.py`](../app/services/threads.py) | thread records in the LangGraph store, one namespace per user |
| [`dependencies.py`](../app/dependencies.py) | `owned_thread` (404), `idle_thread` / `interrupted_thread` (409) |
| [`routers/runs.py`](../app/routers/runs.py) | `POST /threads/{thread_id}/resume` → `Command(resume=…)` |

Docs: [FastAPI OAuth2 with Password and JWT](https://fastapi.tiangolo.com/tutorial/security/oauth2-jwt/) ·
[Dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/) · [jwt.io introduction](https://www.jwt.io/introduction) ·
[LangChain human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) ·
[OWASP API1:2023 BOLA](https://owasp.org/API-Security/editions/2023/en/0xa1-broken-object-level-authorization/)

In [1]:
import logging
import sys

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", stream=sys.stdout)  # log lines in the normal output
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from utils import BASE_URL, PROJECT_DIR, check_server, client, short, short_token, stream_run  # notebooks/utils.py

check_server()

server: ok


## 1. Accounts: the password is stored only as a hash

`create_user` stores `{user_id, name, password_hash}` under the store namespace `("auth", "users")`. The hash is
Argon2 (via `pwdlib`, as the FastAPI tutorial recommends): slow and salted, so a leaked database does not reveal the
passwords. `authenticate` verifies a password against it, and checks a dummy hash when the user does not exist, so
the response time does not tell an attacker which usernames exist.

> **Read in the editor:** [`app/core/security.py`](../app/core/security.py), `create_user`, lines 27–33.

> **Read in the editor:** [`app/core/security.py`](../app/core/security.py), `authenticate`, lines 36–43.

Sign Alice and Bob up **through the running API**, as the app's sign-up page does. Running the notebook again gives
409 (the username is taken), which is fine: the account already exists.

In [2]:
import httpx

for username, name, password in (("alice", "Alice", "alice-pass-2026"), ("bob", "Bob", "bob-pass-2026")):
    r = httpx.post(f"{BASE_URL}/auth/signup", json={"username": username, "password": password, "name": name})
    print(f"signup {username}: {r.status_code}", r.json() if r.status_code == 201 else r.json()["detail"])
r = httpx.post(f"{BASE_URL}/auth/signup", json={"username": "x", "password": "short", "name": "X"})
print("bad signup:", r.status_code, [(e["loc"][-1], e["msg"]) for e in r.json()["errors"]])

signup alice: 409 Username alice is already taken
signup bob: 409 Username bob is already taken
bad signup: 422 [('username', 'String should have at least 3 characters'), ('password', 'String should have at least 8 characters')]


## 2. Sign in: username + password → a signed JWT

`POST /auth/token` follows OAuth2's password flow: the client sends a **form** (`username`, `password`) and receives
`{"access_token": "<JWT>", "token_type": "bearer"}`. Swagger UI's *Authorize* button uses exactly this endpoint.

A JWT has three base64url parts, `header.payload.signature`. The payload carries the **claims**: `sub` (the user),
`exp` (expiry, here 30 minutes). The signature is an HMAC-SHA256 of the first two parts with `JWT_SECRET`: anyone can
read the payload, nobody without the secret can change it.

> **Read in the editor:** [`app/core/security.py`](../app/core/security.py), `create_access_token`, lines 47–52.

In [3]:
r = httpx.post(f"{BASE_URL}/auth/token", data={"username": "alice", "password": "alice-pass-2026"})
body = r.json()
token = body["access_token"]
print(r.status_code, "| token_type:", body["token_type"], "| expires_in:", body["expires_in"], "s")
print("token:", short_token(token), f"({len(token)} characters, three parts: {token.count('.') + 1})")

import jwt  # PyJWT: read the parts without verifying, to see what is inside

print("header: ", jwt.get_unverified_header(token))
claims = jwt.decode(token, options={"verify_signature": False})
print("payload:", {k: claims[k] for k in ("sub", "name")}, "| exp in", claims["exp"] - int(__import__("time").time()), "s")

r = httpx.post(f"{BASE_URL}/auth/token", data={"username": "alice", "password": "wrong-password"})
print("\nwrong password:", r.status_code, r.json()["detail"])

200 | token_type: bearer | expires_in: 1800 s
token: eyJhbGciOiJIUzI1NiIs… (144 characters, three parts: 3)
header:  {'alg': 'HS256', 'typ': 'JWT'}
payload: {'sub': 'alice', 'name': 'Alice'} | exp in 1800 s

wrong password: 401 Incorrect username or password


## 3. Every request: `current_user` verifies the token

`OAuth2PasswordBearer` reads `Authorization: Bearer <token>`; `current_user` verifies signature and expiry with
`jwt.decode(..., algorithms=["HS256"])` and returns the `User`, or raises **401** with `WWW-Authenticate: Bearer`.
Every route that declares `user: CurrentUser` runs it first; the user is never read from the request body.

> **Read in the editor:** [`app/core/security.py`](../app/core/security.py), `current_user`, lines 66–76.

In [4]:
from datetime import datetime, timedelta, timezone

from app.core.config import get_settings

secret = get_settings().jwt_secret.get_secret_value()
expired = jwt.encode({"sub": "alice", "exp": datetime.now(timezone.utc) - timedelta(minutes=1)}, secret, algorithm="HS256")
forged = jwt.encode({"sub": "alice", "exp": datetime.now(timezone.utc) + timedelta(minutes=30)}, "an-attacker-guess-of-the-secret-0000",
                    algorithm="HS256")
cases = [("valid token", token), ("no token", None), ("expired token", expired),
         ("signed with another secret", forged), ("payload edited (sub=bob)", token.split(".")[0] + "."
          + jwt.utils.base64url_encode(b'{"sub":"bob","exp":9999999999}').decode() + "." + token.split(".")[2])]
for label, t in cases:
    r = httpx.get(f"{BASE_URL}/auth/me", headers={"Authorization": f"Bearer {t}"} if t else {})
    print(f"{label:<30}{r.status_code}  {r.json() if r.status_code == 200 else r.json()['detail']}")

valid token                   200  {'user_id': 'alice', 'name': 'Alice'}
no token                      401  Missing, invalid or expired token
expired token                 401  Missing, invalid or expired token
signed with another secret    401  Missing, invalid or expired token
payload edited (sub=bob)      401  Missing, invalid or expired token


## 4. Whose thread: one store namespace per user

A thread record is saved under `("users", user_id, "threads")`. `get_thread` looks an id up **in the caller's
namespace only**, so another user's thread is simply not found. Alice creates a thread through the API; the cell then
reads the service's own store (the SQLite file `output/agent.db`) as Alice and as Bob:

> **Read in the editor:** [`app/services/threads.py`](../app/services/threads.py), `get_thread`, lines 21–23.

In [5]:
import aiosqlite
from langgraph.store.sqlite.aio import AsyncSqliteStore

from app.core.config import get_settings
from app.services import threads

alice, bob = client("alice"), client("bob")
thread_id = alice.post("/threads").json()["thread_id"]  # as the UI does
store = AsyncSqliteStore(await aiosqlite.connect(get_settings().db_path, isolation_level=None))
print("created for alice:", thread_id, "stored under", threads.namespace("alice"))
print("alice looks it up:", await threads.get_thread(store, "alice", thread_id))
print("bob looks it up:  ", await threads.get_thread(store, "bob", thread_id))

created for alice: 492ced1be977 stored under ('users', 'alice', 'threads')
alice looks it up: thread_id='492ced1be977' title='New brief' created_at='2026-11-08T19:48:35+00:00' status='idle'
bob looks it up:   None


`owned_thread` turns "not found" into **404**, the same answer as for an id that does not exist, so ids cannot be
probed. Because it is a dependency, it runs before the route, before an SSE stream sends its `200` and first byte.

> **Read in the editor:** [`app/dependencies.py`](../app/dependencies.py), `owned_thread`, lines 30–34.

## 5. Bob tries every route to Alice's thread

Alice starts a brief on the running service; it pauses for approval. Then Bob tries the direct read **and** the
indirect routes (run, resume, list, a forged `user_id` in the body). Each must fail.

In [6]:
events = stream_run("alice", f"/threads/{thread_id}/runs",
                    {"message": "Brief on how long electric-car batteries last."}, quiet=True)
print("Alice's run:", events[-1]["data"]["status"], "| thread:", alice.get(f"/threads/{thread_id}").json()["status"])

Alice's run: interrupted | thread: interrupted


In [7]:
bob_thread = bob.post("/threads").json()["thread_id"]
attempts = [
    ("GET  Alice's thread and its messages", lambda: bob.get(f"/threads/{thread_id}"), 404),
    ("POST a run on Alice's thread", lambda: bob.post(f"/threads/{thread_id}/runs", json={"message": "hi"}), 404),
    ("POST resume: approve Alice's save_brief", lambda: bob.post(f"/threads/{thread_id}/resume",
                                                               json={"decision": "approve"}), 404),
    ("POST a run with user_id='alice' in the body", lambda: bob.post(f"/threads/{bob_thread}/runs",
                                                                     json={"message": "hi", "user_id": "alice"}), 422),
    ("GET  a thread id that does not exist", lambda: bob.get("/threads/000000000000"), 404),
]
print(f"{'Bob tries to':<46}{'status':>7}  result")
for label, call, expected in attempts:
    r = call()
    print(f"{label:<46}{r.status_code:>7}  {'PASS' if r.status_code == expected else 'FAIL'}  {short(r.json()['detail'], 45)}")
listed = [t["thread_id"] for t in bob.get("/threads").json()]
print(f"{'GET  /threads (his list)':<46}{200:>7}  {'PASS' if thread_id not in listed else 'FAIL'}  "
      f"{len(listed)} thread(s), none of Alice's")
print("\nAlice's thread is still waiting:", alice.get(f"/threads/{thread_id}").json()["status"])

Bob tries to                                   status  result
GET  Alice's thread and its messages              404  PASS  Thread 492ced1be977 not found
POST a run on Alice's thread                      404  PASS  Thread 492ced1be977 not found
POST resume: approve Alice's save_brief           404  PASS  Thread 492ced1be977 not found
POST a run with user_id='alice' in the body       422  PASS  The request body or parameters are invalid.
GET  a thread id that does not exist              404  PASS  Thread 000000000000 not found
GET  /threads (his list)                          200  PASS  5 thread(s), none of Alice's

Alice's thread is still waiting: interrupted


## 6. Human approval over HTTP: resume

`resume_run` accepts only an `interrupted` thread of the caller (`InterruptedThread`, else 409) and turns
`{"decision": ...}` into `Command(resume={"decisions": [...]})`, the format `HumanInTheLoopMiddleware` expects. The
rest of the run streams on a **new** SSE response.

> **Read in the editor:** [`app/routers/runs.py`](../app/routers/runs.py), `resume_run`, lines 30–40.

In [8]:
pending = alice.get(f"/threads/{thread_id}").json()["interrupt"]["action_requests"][0]
print(pending["name"], "| title:", pending["args"]["title"], "\n" + pending["args"]["content"][:300], "…\n")
events = stream_run("alice", f"/threads/{thread_id}/resume", {"decision": "approve"})

save_brief | title: How long EV batteries last 
## Findings
- EV battery life is best understood as gradual capacity loss rather than a fixed replacement date: Geotab reports that degradation varies by vehicle model and is influenced by charging power, climate, and usage. Its analysis estimates an average battery lifespan of around 13 years or mo …

[metadata] run 91e26e23 on thread 492ced1be977
[tool_call] save_brief({"title": "How long EV batteries last", "content": "## Findings\n- EV battery life is best…)
[step] tools
[tool_result] save_brief -> success: saved: briefs/alice/how-long-ev-batteries-last.md
[token] - EV batteries typically lose capacity gradually rather than reaching a fixed end date; Geotab estimates an average lifespan of around 13 years or more, with results varying by model and use.
- Geotab’s 2024 analysis found average degradation of 1.8% per year—an average, not a prediction for every vehicle.
- U.S. battery warranties commonly cover 8 years or 100,000 miles, of

In [9]:
saved = next(e["data"]["preview"] for e in events if e["event"] == "tool_result" and e["data"]["name"] == "save_brief")
path = PROJECT_DIR / "output" / saved.removeprefix("saved: ")
print("file:", path.relative_to(PROJECT_DIR), "| exists:", path.exists())

file: output/briefs/alice/how-long-ev-batteries-last.md | exists: True


`save_brief` wrote under `output/briefs/alice/`: the tool takes the user from `ToolRuntime.context.user_id`, which
the server filled from the token. Now a **reject** with a reason: the tool does not run, the model reads the reason.

In [10]:
reject_thread = alice.post("/threads").json()["thread_id"]
events = stream_run("alice", f"/threads/{reject_thread}/runs",
                    {"message": "Brief on the payback time of rooftop solar panels."}, quiet=True)
print("first run:", events[-1]["data"]["status"])
events = stream_run("alice", f"/threads/{reject_thread}/resume",
                    {"decision": "reject", "message": "Do not save it yet, I will check the sources first."})
print("save_brief ran:", any(e["event"] == "tool_result" and e["data"]["name"] == "save_brief"
                            and e["data"]["status"] == "success" for e in events))

first run: interrupted
[metadata] run 77c504db on thread ea3789181060
[tool_call] save_brief({"title": "Rooftop solar payback", "content": "## Findings\n- For U.S. homeowners, EnergyS…)
[tool_result] save_brief -> error: User rejected the tool call for `save_brief` with reason: Do not save it yet, I …
[token] - EnergySage reports an average U.S. rooftop-solar payback of about 10 years, with estimates ranging from roughly 5 to 15 years.
- Payback depends on factors such as electricity costs, system size, and payment method.
- Export-credit rules vary by state and affect savings; no home-specific estimate is possible without your location, utility plan, and system quote.

**Not saved**—you asked to check the sources first.
[step] model
[done] completed in 2.9 s
save_brief ran: False
